In [ ]:
# Setting system path and project root
import os
import sys

PROJECT_ROOT_DIR = os.path.abspath('../../')
sys.path.append(PROJECT_ROOT_DIR) # bringing system path to project root

def get_fp(relative_path):
    return os.path.join(PROJECT_ROOT_DIR, relative_path)

### Grasp

In [ ]:
from src.const.dataset import KgqaDataset, DatasetSplit
from src.const.llm import ChatModel
from src.util.external_kgqa import evaluate_external_system, qald_to_grasp_jsonl, grasp_output_to_tsv
from src.util.qald_io import _get_gerbil_ready_filepath
from pathlib import Path

In [ ]:
## Dictionary of input dataset and output path

llm_config = ChatModel.GPTOSS120B.value # LLM to use
grasp_info = {
    'qald10_test': {
        'ds': KgqaDataset.QALD10_UPDATED_TENTRISQ10,
        'split' : DatasetSplit.TEST,
        'input_dir': 'data_dir/external_systems/grasp/input/qald10',
        'orig_out_dir': f'data_dir/external_systems/grasp/output/original/{llm_config.model_id}/qald10',
        'tsv_out_dir': f'data_dir/external_systems/grasp/output/tsv/{llm_config.model_id}/qald10',
        'gerbil_out_dir': f'data_dir/external_systems/grasp/output/gerbil/{llm_config.model_id}/qald10',
        'langs': ['en', 'de', 'ru', 'zh']
    },
    'qald9plus_test': {
        'ds': KgqaDataset.QALD9PLUS_UPDATED_TENTRISQ10,
        'split' : DatasetSplit.TEST,
        'input_dir': 'data_dir/external_systems/grasp/input/qald9plus',
        'orig_out_dir': f'data_dir/external_systems/grasp/output/original/{llm_config.model_id}/qald9plus',
        'tsv_out_dir': f'data_dir/external_systems/grasp/output/tsv/{llm_config.model_id}/qald9plus',
        'gerbil_out_dir': f'data_dir/external_systems/grasp/output/gerbil/{llm_config.model_id}/qald9plus',
        'langs': ['en', 'de', 'fr', 'ba', 'be', 'es', 'hy', 'ru', 'uk']
    },
    'lcquad2_test': {
        'ds': KgqaDataset.LCQUAD2_UPDATED_TENTRISQ10,
        'split' : DatasetSplit.TEST,
        'input_dir': 'data_dir/external_systems/grasp/input/lcquad2',
        'orig_out_dir': f'data_dir/external_systems/grasp/output/original/{llm_config.model_id}/lcquad2',
        'tsv_out_dir': f'data_dir/external_systems/grasp/output/tsv/{llm_config.model_id}/lcquad2',
        'gerbil_out_dir': f'data_dir/external_systems/grasp/output/gerbil/{llm_config.model_id}/lcquad2',
        'langs': ['en']
    },
}

In [ ]:
## Convert input QALD to jsonl for GRASP
# For each dataset
for key, ds_info in grasp_info.items():
    print(f'Processing {key}')
    ds_obj = ds_info['ds'].value
    ds_split = ds_info['split']
    ds_langs = ds_info['langs']
    # extract the gold qald file path
    gold_qald_fp = get_fp(ds_obj.split_dict[ds_split])
    jsonl_dir = ds_info['input_dir']
    # Create two jsonl for each language - native, translated
    for lang in ds_langs:
        file_base_name = f'{ds_obj.dataset_id}_{ds_split.name.lower()}.jsonl'
        print(f'Creating jsonl (native lang) for: {lang}')
        jsonl_file_name = f'{lang}_native_{file_base_name}'
        jsonl_file_path = os.path.join(jsonl_dir, jsonl_file_name)
        qald_to_grasp_jsonl(gold_qald_fp, jsonl_file_path, lang, use_translation=False)
        if lang != "en":
            print(f'Creating jsonl (translated) for: {lang}')
            jsonl_file_name = f'{lang}_translated_{file_base_name}'
            jsonl_file_path = os.path.join(jsonl_dir, jsonl_file_name)
            qald_to_grasp_jsonl(gold_qald_fp, jsonl_file_path, lang, use_translation=True)

In [ ]:
## Convert output of GRASP to TSV to be processed
# TODO: Obsolete, update this logic
# tsv_file_path = f'{output_dir}/tsv/gpt-oss-120b/qald9plus_tentrisq10_train_output.tsv'
# grasp_output_to_tsv(f'{output_dir}/original/gpt-oss-120b/qald9plus_tentrisq10_train_output.jsonl', tsv_file_path, True, llm_config)

In [ ]:
## Generate QALD json and execute gerbil experiment
for key, ds_info in grasp_info.items():
    print(f'Processing {key}')
    ds_obj = ds_info['ds'].value
    ds_split = ds_info['split']
    # extract the gold qald file path
    gold_qald_fp = get_fp(ds_obj.split_dict[ds_split])
    gerbilready_gold_json_path = _get_gerbil_ready_filepath(gold_qald_fp)
    tsv_out_dir = get_fp(ds_info['tsv_out_dir'])
    gerbil_out_dir = get_fp(ds_info['gerbil_out_dir'])
    # for each language qald json in 'orig_out_dir'
    for file_path in Path(tsv_out_dir).rglob("*.tsv"):
        # fetch filepath
        if file_path.is_file():
            print(f'Evaluating {file_path}')
            lang_id = os.path.splitext(os.path.basename(file_path))[0]
            sysname = f'grasp-{key}-{lang_id}'
            evaluate_external_system(sysname, ds_info['ds'], ds_split, gerbilready_gold_json_path, str(file_path), gerbil_out_dir, 'en')

### MST5

In [ ]:
from src.const.dataset import KgqaDataset, DatasetSplit
from src.const.llm import ChatModel
from src.util.external_kgqa import evaluate_external_system, generate_qald_output_tsv
from src.util.qald_io import _get_gerbil_ready_filepath
from pathlib import Path

In [ ]:
## Dictionary of input dataset and output path
mst5_info = {
    'qald10_test': {
        'ds': KgqaDataset.QALD10_UPDATED_TENTRISQ10,
        'split' : DatasetSplit.TEST,
        'orig_out_dir': get_fp('data_dir/external_systems/mst5/output/original/qald10'),
        'tsv_out_dir': get_fp('data_dir/external_systems/mst5/output/tsv/qald10'),
        'gerbil_out_dir': get_fp('data_dir/external_systems/mst5/output/gerbil/qald10')
    },
    'qald9plus_test': {
        'ds': KgqaDataset.QALD9PLUS_UPDATED_TENTRISQ10,
        'split' : DatasetSplit.TEST,
        'orig_out_dir': get_fp('data_dir/external_systems/mst5/output/original/qald9plus'),
        'tsv_out_dir': get_fp('data_dir/external_systems/mst5/output/tsv/qald9plus'),
        'gerbil_out_dir': get_fp('data_dir/external_systems/mst5/output/gerbil/qald9plus')
    },
}

In [ ]:
## Converting MST5 Qald files to tsvs for streamlined evaluation
# for each entry in info
for key, ds_info in mst5_info.items():
    print(f'Processing {key}')
    ds_obj = ds_info['ds'].value
    ds_split = ds_info['split']
    # extract the gold qald file path
    gold_qald_fp = get_fp(ds_obj.split_dict[ds_split])
    qald_out_dir = ds_info['orig_out_dir']
    tsv_out_dir = ds_info['tsv_out_dir']
    # for each language qald json in 'orig_out_dir'
    for file_path in Path(qald_out_dir).rglob("*"):
        # fetch filepath
        if file_path.is_file():
            # create tsv out path
            lang_id = os.path.splitext(os.path.basename(file_path))[0]
            tsv_out_path = os.path.join(tsv_out_dir, f'{lang_id}.tsv')
            # call generate_mst5_output_tsv
            # print(tsv_out_path)
            generate_qald_output_tsv(gold_qald_fp, str(file_path), tsv_out_path)

In [ ]:
## Evaluate generated TSVs
for key, ds_info in mst5_info.items():
    print(f'Processing {key}')
    ds_obj = ds_info['ds'].value
    ds_split = ds_info['split']
    # extract the gold qald file path
    gold_qald_fp = get_fp(ds_obj.split_dict[ds_split])
    gerbilready_gold_json_path = _get_gerbil_ready_filepath(gold_qald_fp)
    tsv_out_dir = ds_info['tsv_out_dir']
    gerbil_out_dir = ds_info['gerbil_out_dir']
    # for each language qald json in 'orig_out_dir'
    for file_path in Path(tsv_out_dir).rglob("*"):
        # fetch filepath
        if file_path.is_file():
            lang_id = os.path.splitext(os.path.basename(file_path))[0]
            sysname = f'mst5-{key}-{lang_id}'
            evaluate_external_system(sysname, ds_info['ds'], ds_split, gerbilready_gold_json_path, str(file_path), gerbil_out_dir, 'en')

### UniQ-Gen

In [ ]:
from src.const.dataset import KgqaDataset, DatasetSplit
from src.const.llm import ChatModel
from src.util.external_kgqa import evaluate_external_system, generate_qald_output_tsv
from src.util.qald_io import _get_gerbil_ready_filepath
from pathlib import Path

In [ ]:
## Dictionary of input dataset and output path
uniqgen_info = {
    'lcquad2_test': {
        'ds': KgqaDataset.LCQUAD2_UPDATED_TENTRISQ10,
        'split' : DatasetSplit.TEST,
        'orig_out_dir': get_fp('data_dir/external_systems/uniqgen/output/original/lcquad2'),
        'tsv_out_dir': get_fp('data_dir/external_systems/uniqgen/output/tsv/lcquad2'),
        'gerbil_out_dir': get_fp('data_dir/external_systems/uniqgen/output/gerbil/lcquad2')
    },
}

In [ ]:
## Converting Qald files to tsvs for streamlined evaluation
# for each entry in info
for key, ds_info in uniqgen_info.items():
    print(f'Processing {key}')
    ds_obj = ds_info['ds'].value
    ds_split = ds_info['split']
    # extract the gold qald file path
    gold_qald_fp = get_fp(ds_obj.split_dict[ds_split])
    qald_out_dir = ds_info['orig_out_dir']
    tsv_out_dir = ds_info['tsv_out_dir']
    # for each language qald json in 'orig_out_dir'
    for file_path in Path(qald_out_dir).rglob("*"):
        # fetch filepath
        if file_path.is_file():
            # create tsv out path
            lang_id = os.path.splitext(os.path.basename(file_path))[0]
            tsv_out_path = os.path.join(tsv_out_dir, f'{lang_id}.tsv')
            # call generate_mst5_output_tsv
            # print(tsv_out_path)
            generate_qald_output_tsv(gold_qald_fp, str(file_path), tsv_out_path)

In [ ]:
## Evaluate generated TSVs
for key, ds_info in uniqgen_info.items():
    print(f'Processing {key}')
    ds_obj = ds_info['ds'].value
    ds_split = ds_info['split']
    # extract the gold qald file path
    gold_qald_fp = get_fp(ds_obj.split_dict[ds_split])
    gerbilready_gold_json_path = _get_gerbil_ready_filepath(gold_qald_fp)
    tsv_out_dir = ds_info['tsv_out_dir']
    gerbil_out_dir = ds_info['gerbil_out_dir']
    # for each language qald json in 'orig_out_dir'
    for file_path in Path(tsv_out_dir).rglob("*"):
        # fetch filepath
        if file_path.is_file():
            lang_id = os.path.splitext(os.path.basename(file_path))[0]
            sysname = f'uniqgen-{key}-{lang_id}'
            evaluate_external_system(sysname, ds_info['ds'], ds_split, gerbilready_gold_json_path, str(file_path), gerbil_out_dir, 'en')